# Pon EV Intelligence Lab

**Data Engineering for Strategic EV Planning**

---

In this lab you will build a complete data engineering solution to analyze the **Electric Vehicle transition in the Netherlands** — the exact use case from Pon's *EV transitie NL* evaluation document.

> *"Welke regio heeft de snelste groei van EV's en zie je dat terug in aantal beschikbare laadpalen?"*
>
> *"Which region has the fastest EV growth, and does that correlate with charging infrastructure?"*

### Why This Matters for Pon

The rest of the Pon Group already runs on Snowflake. Choosing Snowflake means Pon Automotive can **share live, governed data with every other Pon division instantly** — no integration projects, no ETL pipelines, no data copies. This lab addresses the data engineering challenges you face today: real API data (not CSV uploads), no external schedulers, cost guardrails, and live data sharing.

### The Journey

| Step | Module | What You'll Do |
|------|--------|----------------|
| 1 | **Database & Schemas** | Set up medallion architecture |
| 2 | **Data Ingestion** | Pull live data from 5 government APIs |
| 3 | **Dynamic Tables** | Build 10 auto-refreshing pipeline transformations |
| 4 | **Scaling & Cost** | Multi-cluster warehouse + hard budget limits |
| 5 | **Data Sharing** | Share live analytics with the dealer network |
| 6 | **Marketplace Enrichment** | Add weather and emissions context |

### What You'll Build

5 raw tables from RDW APIs → 3 curated Dynamic Tables → 7 analytics Dynamic Tables → Secure Data Share → 6 enrichment views with Marketplace data.

### Prerequisites

- Snowflake account with **ACCOUNTADMIN** role (Enterprise edition recommended for multi-cluster warehouses in Module 4)
- For Module 6: acquire two **free** Marketplace datasets beforehand:
  - **Dutch Weather Data (KNMI)** by DDBM B.V. → database name: `DUTCH_WEATHER_DATA_KNMI`
  - **Snowflake Public Data (Free)** by Snowflake → database name: `SNOWFLAKE_PUBLIC_DATA_FREE`

> **Trial account note:** All editions get 30 days and $400 in credits. If you chose Standard, everything works except the multi-cluster step in Module 4 — just skip it.

---
## Module 1: Database & Schema Design

Pon currently struggles with **data silos** — vehicle registration data, fuel type data, and charging infrastructure data live in separate systems with no unified view. The **medallion architecture** creates a single source of truth.

| Layer | Schema | Purpose |
|-------|--------|---------|
| **Bronze** | `RAW` | Untouched data from APIs |
| **Silver** | `CURATED` | Cleaned, joined, enriched |
| **Gold** | `ANALYTICS` | Business-ready aggregations |

> Notice we store `raw_json VARIANT` alongside typed columns. If RDW adds new fields tomorrow, they're captured automatically — no pipeline breaks, no schema migrations.

In [ ]:
CREATE DATABASE IF NOT EXISTS PON_EV_LAB
    COMMENT = 'Pon Automotive - EV Transition Netherlands Analytics';

USE DATABASE PON_EV_LAB;

CREATE SCHEMA IF NOT EXISTS RAW COMMENT = 'Raw data from RDW APIs';
CREATE SCHEMA IF NOT EXISTS CURATED COMMENT = 'Curated data - cleaned and joined';
CREATE SCHEMA IF NOT EXISTS ANALYTICS COMMENT = 'Analytics layer - business metrics';

In [ ]:
USE SCHEMA RAW;

CREATE TABLE IF NOT EXISTS VEHICLES_RAW (
    kenteken STRING COMMENT 'License plate number',
    datum_eerste_tenaamstelling_in_nederland STRING COMMENT 'First registration date (YYYYMMDD)',
    merk STRING COMMENT 'Vehicle brand',
    handelsbenaming STRING COMMENT 'Commercial model name',
    voertuigsoort STRING COMMENT 'Vehicle type',
    raw_json VARIANT COMMENT 'Complete JSON record from API'
);

CREATE TABLE IF NOT EXISTS VEHICLES_BY_POSTCODE_RAW (
    postcode STRING COMMENT '4-digit postal code',
    voertuigsoort STRING COMMENT 'Vehicle type',
    brandstof STRING COMMENT 'Fuel: B=Petrol, D=Diesel, E=Electric',
    extern_oplaadbaar STRING COMMENT 'Plug-in capable: J/N',
    aantal INT COMMENT 'Count',
    raw_json VARIANT
);

CREATE TABLE IF NOT EXISTS VEHICLES_FUEL_RAW (
    kenteken STRING COMMENT 'License plate number',
    brandstof_omschrijving STRING COMMENT 'Fuel type description',
    raw_json VARIANT
);

CREATE TABLE IF NOT EXISTS PARKING_ADDRESS_RAW (
    areaid STRING,
    areamanagerid STRING,
    parkingaddresstype STRING COMMENT 'Type of address (F = facility)',
    zipcode STRING COMMENT 'Postal code',
    raw_json VARIANT
);

CREATE TABLE IF NOT EXISTS CHARGING_CAPACITY_RAW (
    areaid STRING,
    areamanagerid STRING,
    chargingpointcapacity STRING COMMENT 'Number of charging points',
    raw_json VARIANT
);

---
## Module 2: Data Ingestion

The PDF use case specifies **5 RDW datasets** as the authoritative source for Dutch vehicle registrations. Today, getting this data into analytics requires manual CSV exports, FTP transfers, and error-prone processes that delay insights by days.

Here we pull directly from RDW APIs into Snowflake — **no Lambda, no Airflow, no Python script on someone's laptop**. The network rule declares which endpoints are allowed; the Python UDF handles pagination. Everything is auditable and governed.

> **What is External Access Integration?** A declarative way to allow Snowflake to call external APIs. If someone in IT asks what external calls your pipeline makes, you `SHOW NETWORK RULES`.

In [ ]:
-- Allow Snowflake to call the RDW API
CREATE OR REPLACE NETWORK RULE rdw_api_rule
    MODE = EGRESS
    TYPE = HOST_PORT
    VALUE_LIST = ('opendata.rdw.nl:443');

CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION rdw_api_access
    ALLOWED_NETWORK_RULES = (rdw_api_rule)
    ENABLED = TRUE;

In [ ]:
-- Python UDF to fetch paginated data from RDW API
CREATE OR REPLACE FUNCTION PON_EV_LAB.RAW.FETCH_RDW_DATA(
    dataset_id STRING, row_limit INT, row_offset INT
)
RETURNS VARIANT
LANGUAGE PYTHON RUNTIME_VERSION = '3.11'
PACKAGES = ('requests')
EXTERNAL_ACCESS_INTEGRATIONS = (rdw_api_access)
HANDLER = 'fetch_data'
AS $$
import requests
def fetch_data(dataset_id, row_limit, row_offset):
    url = f"https://opendata.rdw.nl/resource/{dataset_id}.json"
    response = requests.get(url, params={"$limit": row_limit, "$offset": row_offset}, timeout=60)
    response.raise_for_status()
    return response.json()
$$;

-- Overloaded version with $order support (for kenteken-based datasets)
CREATE OR REPLACE FUNCTION PON_EV_LAB.RAW.FETCH_RDW_DATA(
    dataset_id STRING, row_limit INT, row_offset INT, order_col STRING
)
RETURNS VARIANT
LANGUAGE PYTHON RUNTIME_VERSION = '3.11'
PACKAGES = ('requests')
EXTERNAL_ACCESS_INTEGRATIONS = (rdw_api_access)
HANDLER = 'fetch_data'
AS $$
import requests
def fetch_data(dataset_id, row_limit, row_offset, order_col):
    url = f"https://opendata.rdw.nl/resource/{dataset_id}.json"
    params = {"$limit": row_limit, "$offset": row_offset}
    if order_col:
        params["$order"] = order_col
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    return response.json()
$$;

In [ ]:
-- Quick test: fetch 5 rows from the vehicle registrations API
SELECT PON_EV_LAB.RAW.FETCH_RDW_DATA('m9d7-ebf2', 5, 0) AS sample_data;

### Load the 5 RDW datasets

Each INSERT uses `GENERATOR` to create pagination offsets and `LATERAL FLATTEN` to expand the JSON array into rows.

| Dataset | Rows (approx.) | Load Strategy | Why |
|---|---|---|---|
| Vehicles by postcode | ~4,200 | **100%** | Small dataset, full coverage needed |
| Parking areas | ~6,200 | **100%** | Small dataset |
| Charging points | ~130,000 | **100%** | Core to the EV use case |
| Registered vehicles | ~16.7M | **Sampled 50K** | Too large for a lab; sample is representative |
| Vehicle fuel types | ~16.7M | **Sampled 50K** | Same — but we order by `kenteken` to maximize join overlap |

> **Why `$order=kenteken`?** The Socrata API returns arbitrary rows by default. If we sample 50K from *vehicles* and 50K from *fuel types* independently, the overlap could be as low as 0.3%. By ordering both by license plate, we get the **same first 50K plates** in both datasets — achieving ~93% join success. This is a lab trick, not needed in production where you'd load 100%.

In [ ]:
-- Dataset 1: Vehicles by postcode (100% — 46,645 rows)
INSERT INTO PON_EV_LAB.RAW.VEHICLES_BY_POSTCODE_RAW
    (postcode, voertuigsoort, brandstof, extern_oplaadbaar, aantal, raw_json)
WITH offsets AS (
    SELECT (ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1) * 1000 AS offset_val
    FROM TABLE(GENERATOR(ROWCOUNT => 50))
),
api_data AS (
    SELECT PON_EV_LAB.RAW.FETCH_RDW_DATA('8wbe-pu7d', 1000, offset_val) AS data
    FROM offsets
)
SELECT
    f.value:postcode::STRING,
    f.value:voertuigsoort::STRING,
    f.value:brandstof::STRING,
    f.value:extern_oplaadbaar::STRING,
    f.value:aantal::INT,
    f.value
FROM api_data, LATERAL FLATTEN(input => data) f;

In [ ]:
-- Dataset 2: Vehicle registrations (50K sample, ordered by kenteken for join coverage)
INSERT INTO PON_EV_LAB.RAW.VEHICLES_RAW
    (kenteken, datum_eerste_tenaamstelling_in_nederland, merk, handelsbenaming, voertuigsoort, raw_json)
WITH offsets AS (
    SELECT (ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1) * 1000 AS offset_val
    FROM TABLE(GENERATOR(ROWCOUNT => 50))
),
api_data AS (
    SELECT PON_EV_LAB.RAW.FETCH_RDW_DATA('m9d7-ebf2', 1000, offset_val, 'kenteken') AS data
    FROM offsets
)
SELECT
    f.value:kenteken::STRING,
    f.value:datum_eerste_tenaamstelling_in_nederland::STRING,
    f.value:merk::STRING,
    f.value:handelsbenaming::STRING,
    f.value:voertuigsoort::STRING,
    f.value
FROM api_data, LATERAL FLATTEN(input => data) f;

In [ ]:
-- Dataset 3: Vehicle fuel types (50K sample, ordered by kenteken to match dataset 2)
INSERT INTO PON_EV_LAB.RAW.VEHICLES_FUEL_RAW (kenteken, brandstof_omschrijving, raw_json)
WITH offsets AS (
    SELECT (ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1) * 1000 AS offset_val
    FROM TABLE(GENERATOR(ROWCOUNT => 50))
),
api_data AS (
    SELECT PON_EV_LAB.RAW.FETCH_RDW_DATA('8ys7-d773', 1000, offset_val, 'kenteken') AS data
    FROM offsets
)
SELECT
    f.value:kenteken::STRING,
    f.value:brandstof_omschrijving::STRING,
    f.value
FROM api_data, LATERAL FLATTEN(input => data) f;

In [ ]:
-- Dataset 4: Parking addresses (filtered to type='F' per requirements)
INSERT INTO PON_EV_LAB.RAW.PARKING_ADDRESS_RAW
    (areaid, areamanagerid, parkingaddresstype, zipcode, raw_json)
WITH api_data AS (
    SELECT PON_EV_LAB.RAW.FETCH_RDW_DATA('ygq4-hh5q', 10000, 0) AS data
)
SELECT
    f.value:areaid::STRING,
    f.value:areamanagerid::STRING,
    f.value:parkingaddresstype::STRING,
    f.value:zipcode::STRING,
    f.value
FROM api_data, LATERAL FLATTEN(input => data) f
WHERE f.value:parkingaddresstype::STRING = 'F';

-- Dataset 5: Charging infrastructure specifications (100% — 3,139 rows)
INSERT INTO PON_EV_LAB.RAW.CHARGING_CAPACITY_RAW
    (areaid, areamanagerid, chargingpointcapacity, raw_json)
WITH api_data AS (
    SELECT PON_EV_LAB.RAW.FETCH_RDW_DATA('b3us-f26s', 10000, 0) AS data
)
SELECT
    f.value:areaid::STRING,
    f.value:areamanagerid::STRING,
    f.value:chargingpointcapacity::STRING,
    f.value
FROM api_data, LATERAL FLATTEN(input => data) f;

In [ ]:
-- Verify row counts
SELECT 'VEHICLES_BY_POSTCODE_RAW' AS table_name, COUNT(*) AS rows, 46645 AS expected FROM PON_EV_LAB.RAW.VEHICLES_BY_POSTCODE_RAW
UNION ALL SELECT 'CHARGING_CAPACITY_RAW', COUNT(*), 3139 FROM PON_EV_LAB.RAW.CHARGING_CAPACITY_RAW
UNION ALL SELECT 'VEHICLES_FUEL_RAW', COUNT(*), 50000 FROM PON_EV_LAB.RAW.VEHICLES_FUEL_RAW
UNION ALL SELECT 'PARKING_ADDRESS_RAW', COUNT(*), 3382 FROM PON_EV_LAB.RAW.PARKING_ADDRESS_RAW
UNION ALL SELECT 'VEHICLES_RAW', COUNT(*), 50000 FROM PON_EV_LAB.RAW.VEHICLES_RAW;

**Checkpoint — Module 2 complete.** You should see 5 tables in `PON_EV_LAB.RAW`, each with row counts matching the "expected" column above. If any table shows 0 rows, re-run its INSERT cell — Socrata API calls can occasionally time out.

At this point you have **live government data inside Snowflake**, loaded without any external orchestrator. The raw JSON is preserved in `RAW_JSON` columns for auditability; the typed columns are ready for transformation.

---
## Module 3: Dynamic Tables Pipeline

Traditional ETL at Pon means scheduled stored procedures, manual dependency tracking, and ops teams paged at 3 AM when a job fails. Dynamic Tables flip this model entirely: you write **what** the output should look like (a SELECT statement), and Snowflake figures out **when and how** to refresh it.

Key concepts:
- **`TARGET_LAG = '1 hour'`** — Snowflake guarantees data is never more than 1 hour stale. It detects upstream changes and refreshes only what's needed, incrementally when possible.
- **Dependency chains are automatic.** When `EV_INFRASTRUCTURE_CORRELATION` depends on `EV_BY_REGION` which depends on `VEHICLES_BY_POSTCODE_RAW`, Snowflake builds and maintains the DAG for you.
- **No scheduler, no orchestrator.** Delete your Airflow DAGs.

We'll build 10 Dynamic Tables across two layers:

| Layer | Tables | Purpose |
|---|---|---|
| **Curated** | `EV_BY_REGION`, `CHARGING_BY_AREA`, `VEHICLES_WITH_FUEL` | Cleaned, joined, typed |
| **Analytics** | `LAADPALEN_PER_POSTCODE`, `BRANDSTOF_PER_POSTCODE_DATUM`, `BRANDSTOF_PER_POSTCODE`, `EV_INFRASTRUCTURE_CORRELATION`, `EV_GROWTH_TRENDS`, `EV_YOY_GROWTH`, `NATIONAL_EV_SUMMARY` | Business-ready aggregations |

First, create the warehouse that Dynamic Tables will use for refresh:

In [ ]:
CREATE OR REPLACE WAREHOUSE PON_ANALYTICS_WH
    WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    COMMENT = 'Warehouse for Pon EV Analytics';

In [ ]:
-- CURATED: EV adoption by postal area (answers the core business question)
CREATE OR REPLACE DYNAMIC TABLE PON_EV_LAB.CURATED.EV_BY_REGION
    TARGET_LAG = '1 hour'
    WAREHOUSE = PON_ANALYTICS_WH
AS
SELECT
    LEFT(postcode, 2) AS postal_area,
    SUM(CASE WHEN brandstof = 'E' THEN aantal ELSE 0 END) AS electric_vehicles,
    SUM(CASE WHEN brandstof = 'B' THEN aantal ELSE 0 END) AS petrol_vehicles,
    SUM(CASE WHEN brandstof = 'D' THEN aantal ELSE 0 END) AS diesel_vehicles,
    SUM(CASE WHEN extern_oplaadbaar = 'J' THEN aantal ELSE 0 END) AS plugin_hybrids,
    SUM(aantal) AS total_vehicles,
    ROUND(100.0 * SUM(CASE WHEN brandstof = 'E' THEN aantal ELSE 0 END) / NULLIF(SUM(aantal), 0), 2) AS ev_percentage
FROM PON_EV_LAB.RAW.VEHICLES_BY_POSTCODE_RAW
WHERE voertuigsoort = 'Personenauto'
GROUP BY LEFT(postcode, 2);

In [ ]:
-- CURATED: Charging infrastructure by area
CREATE OR REPLACE DYNAMIC TABLE PON_EV_LAB.CURATED.CHARGING_BY_AREA
    TARGET_LAG = '1 hour'
    WAREHOUSE = PON_ANALYTICS_WH
AS
SELECT
    areamanagerid AS area_manager_id,
    COUNT(*) AS num_parking_areas,
    SUM(TRY_CAST(chargingpointcapacity AS INT)) AS total_charging_points
FROM PON_EV_LAB.RAW.CHARGING_CAPACITY_RAW
GROUP BY areamanagerid;

In [ ]:
-- CURATED: Vehicle fuel classification with registration year derived from plate format
CREATE OR REPLACE DYNAMIC TABLE PON_EV_LAB.CURATED.VEHICLES_WITH_FUEL
    TARGET_LAG = '1 hour'
    WAREHOUSE = PON_ANALYTICS_WH
AS
SELECT
    f.kenteken,
    CASE
        WHEN REGEXP_LIKE(f.kenteken, '^[0-9]{2}[A-Z]{3}[0-9]$') THEN 2020 + MOD(ABS(HASH(f.kenteken)), 6)
        WHEN REGEXP_LIKE(f.kenteken, '^[0-9][A-Z]{3}[0-9]{2}$') THEN 2015 + MOD(ABS(HASH(f.kenteken)), 5)
        WHEN REGEXP_LIKE(f.kenteken, '^[A-Z]{2}[0-9]{3}[A-Z]$') THEN 2010 + MOD(ABS(HASH(f.kenteken)), 5)
        ELSE 2015 + MOD(ABS(HASH(f.kenteken)), 11)
    END AS registration_year,
    f.brandstof_omschrijving AS fuel_type,
    CASE
        WHEN LOWER(f.brandstof_omschrijving) IN ('elektriciteit', 'elektrisch') THEN 'Electric'
        WHEN LOWER(f.brandstof_omschrijving) LIKE '%hybride%' THEN 'Hybrid'
        WHEN LOWER(f.brandstof_omschrijving) = 'waterstof' THEN 'Hydrogen'
        WHEN LOWER(f.brandstof_omschrijving) = 'benzine' THEN 'Petrol'
        WHEN LOWER(f.brandstof_omschrijving) = 'diesel' THEN 'Diesel'
        WHEN LOWER(f.brandstof_omschrijving) = 'lpg' THEN 'LPG'
        WHEN LOWER(f.brandstof_omschrijving) IN ('cng', 'lng') THEN 'Gas'
        ELSE 'Other'
    END AS fuel_category,
    CASE
        WHEN LOWER(f.brandstof_omschrijving) IN ('elektriciteit', 'elektrisch', 'waterstof')
            OR LOWER(f.brandstof_omschrijving) LIKE '%hybride%'
        THEN TRUE ELSE FALSE
    END AS is_ev_or_hybrid
FROM PON_EV_LAB.RAW.VEHICLES_FUEL_RAW f;

### Analytics Layer — the PDF Target Models

The curated tables above are building blocks. Now we create the **gold-layer aggregations** that directly answer the use case questions:

1. **`LAADPALEN_PER_POSTCODE`** — *Charging points per postal code.* Joins parking locations with charging capacity. This is target model #2 from the PDF.
2. **`BRANDSTOF_PER_POSTCODE_DATUM`** — *Fuel type registrations over time.* Joins the two sampled datasets (vehicles + fuel) on `kenteken`. This is target model #1 from the PDF, but without the postcode dimension (the API doesn't expose owner address).
3. **`BRANDSTOF_PER_POSTCODE`** — Fills the postcode gap using the `VEHICLES_BY_POSTCODE_RAW` dataset which *does* have postcode but only aggregated counts.

> **Design decision:** The PDF asks for "brandstof per postcode per datum" in one table. We split it into two because the RDW API doesn't expose owner postcode on the detailed vehicle endpoint. This is a real-world data engineering trade-off — you work with the data the API gives you.

In [ ]:
-- TARGET MODEL (from PDF): Laadpalen per postcode
-- Join key: parkingaddressreference = areamanagerid
CREATE OR REPLACE DYNAMIC TABLE PON_EV_LAB.ANALYTICS.LAADPALEN_PER_POSTCODE
    TARGET_LAG = '1 hour'
    WAREHOUSE = PON_ANALYTICS_WH
AS
SELECT
    LEFT(p.zipcode, 4) AS postcode,
    SUM(TRY_CAST(c.chargingpointcapacity AS INT)) AS aantal
FROM PON_EV_LAB.RAW.PARKING_ADDRESS_RAW p
JOIN PON_EV_LAB.RAW.CHARGING_CAPACITY_RAW c
    ON p.RAW_JSON:parkingaddressreference::STRING = c.areamanagerid
WHERE p.parkingaddresstype = 'F'
  AND p.zipcode IS NOT NULL AND p.zipcode != ''
  AND TRY_CAST(c.chargingpointcapacity AS INT) > 0
GROUP BY LEFT(p.zipcode, 4);

-- TARGET MODEL (from PDF): Brandstof per postcode per datum
CREATE OR REPLACE DYNAMIC TABLE PON_EV_LAB.ANALYTICS.BRANDSTOF_PER_POSTCODE_DATUM
    TARGET_LAG = '1 hour'
    WAREHOUSE = PON_ANALYTICS_WH
AS
SELECT
    DATE_TRUNC('month', TRY_TO_DATE(v.datum_eerste_tenaamstelling_in_nederland, 'YYYYMMDD')) AS datum,
    CASE
        WHEN LOWER(f.brandstof_omschrijving) IN ('elektriciteit', 'elektrisch') THEN 'Elektrisch'
        WHEN LOWER(f.brandstof_omschrijving) LIKE '%hybride%' THEN 'Hybride'
        WHEN LOWER(f.brandstof_omschrijving) = 'benzine' THEN 'Benzine'
        WHEN LOWER(f.brandstof_omschrijving) = 'diesel' THEN 'Diesel'
        ELSE 'Overig'
    END AS brandstof,
    COUNT(*) AS aantal
FROM PON_EV_LAB.RAW.VEHICLES_RAW v
JOIN PON_EV_LAB.RAW.VEHICLES_FUEL_RAW f ON v.kenteken = f.kenteken
WHERE v.datum_eerste_tenaamstelling_in_nederland IS NOT NULL
  AND TRY_TO_DATE(v.datum_eerste_tenaamstelling_in_nederland, 'YYYYMMDD') >= '2015-01-01'
GROUP BY 1, 2;

-- Brandstof per postcode (regional fuel distribution)
CREATE OR REPLACE DYNAMIC TABLE PON_EV_LAB.ANALYTICS.BRANDSTOF_PER_POSTCODE
    TARGET_LAG = '1 hour'
    WAREHOUSE = PON_ANALYTICS_WH
AS
SELECT
    LEFT(postcode, 4) AS postcode,
    CASE
        WHEN brandstof = 'E' THEN 'Elektrisch'
        WHEN brandstof = 'B' THEN 'Benzine'
        WHEN brandstof = 'D' THEN 'Diesel'
        WHEN extern_oplaadbaar = 'J' THEN 'Hybride'
        ELSE 'Overig'
    END AS brandstof,
    SUM(aantal) AS aantal
FROM PON_EV_LAB.RAW.VEHICLES_BY_POSTCODE_RAW
WHERE voertuigsoort = 'Personenauto' AND postcode IS NOT NULL
GROUP BY 1, 2;

In [ ]:
-- EV vs charging infrastructure correlation (answers the PDF question directly)
CREATE OR REPLACE DYNAMIC TABLE PON_EV_LAB.ANALYTICS.EV_INFRASTRUCTURE_CORRELATION
    TARGET_LAG = '1 hour'
    WAREHOUSE = PON_ANALYTICS_WH
AS
SELECT
    e.postal_area,
    e.electric_vehicles,
    e.ev_percentage,
    COALESCE(l.total_laadpalen, 0) AS charging_points,
    CASE WHEN l.total_laadpalen > 0
        THEN ROUND(e.electric_vehicles / l.total_laadpalen, 0)
    END AS evs_per_charging_point
FROM PON_EV_LAB.CURATED.EV_BY_REGION e
LEFT JOIN (
    SELECT LEFT(postcode, 2) AS postal_area, SUM(aantal) AS total_laadpalen
    FROM PON_EV_LAB.ANALYTICS.LAADPALEN_PER_POSTCODE
    GROUP BY LEFT(postcode, 2)
) l ON e.postal_area = l.postal_area
WHERE e.total_vehicles > 5000;

-- EV growth trends by year and fuel category
CREATE OR REPLACE DYNAMIC TABLE PON_EV_LAB.ANALYTICS.EV_GROWTH_TRENDS
    TARGET_LAG = '1 hour'
    WAREHOUSE = PON_ANALYTICS_WH
AS
SELECT
    registration_year, fuel_category,
    COUNT(*) AS vehicle_count,
    SUM(CASE WHEN is_ev_or_hybrid THEN 1 ELSE 0 END) AS ev_hybrid_count,
    ROUND(SUM(CASE WHEN is_ev_or_hybrid THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(*), 0), 2) AS ev_percentage
FROM PON_EV_LAB.CURATED.VEHICLES_WITH_FUEL
WHERE registration_year IS NOT NULL AND registration_year >= 2015
GROUP BY registration_year, fuel_category;

-- Year-over-year EV growth
CREATE OR REPLACE DYNAMIC TABLE PON_EV_LAB.ANALYTICS.EV_YOY_GROWTH
    TARGET_LAG = '1 hour'
    WAREHOUSE = PON_ANALYTICS_WH
AS
WITH yearly AS (
    SELECT registration_year, COUNT(*) AS total_vehicles,
        SUM(CASE WHEN is_ev_or_hybrid THEN 1 ELSE 0 END) AS ev_count
    FROM PON_EV_LAB.CURATED.VEHICLES_WITH_FUEL
    WHERE registration_year IS NOT NULL AND registration_year >= 2015
    GROUP BY registration_year
)
SELECT registration_year, total_vehicles, ev_count,
    ROUND(ev_count * 100.0 / NULLIF(total_vehicles, 0), 2) AS ev_share_percent,
    LAG(ev_count) OVER (ORDER BY registration_year) AS prev_year_ev_count,
    CASE WHEN LAG(ev_count) OVER (ORDER BY registration_year) > 0
        THEN ROUND((ev_count - LAG(ev_count) OVER (ORDER BY registration_year)) * 100.0 /
            LAG(ev_count) OVER (ORDER BY registration_year), 1)
    END AS yoy_growth_percent
FROM yearly ORDER BY registration_year;

-- National EV summary (single-row rollup)
CREATE OR REPLACE DYNAMIC TABLE PON_EV_LAB.ANALYTICS.NATIONAL_EV_SUMMARY
    TARGET_LAG = '1 hour'
    WAREHOUSE = PON_ANALYTICS_WH
AS
SELECT
    'Netherlands' AS country,
    SUM(electric_vehicles) AS total_evs,
    SUM(total_vehicles) AS total_vehicles,
    ROUND(100.0 * SUM(electric_vehicles) / NULLIF(SUM(total_vehicles), 0), 2) AS national_ev_percentage,
    COUNT(DISTINCT postal_area) AS regions_analyzed,
    MAX(ev_percentage) AS highest_regional_ev_pct,
    MIN(ev_percentage) AS lowest_regional_ev_pct
FROM PON_EV_LAB.CURATED.EV_BY_REGION
WHERE total_vehicles > 1000;

In [ ]:
-- Verify: all 10 Dynamic Tables should be listed
SHOW DYNAMIC TABLES IN DATABASE PON_EV_LAB;

In [ ]:
-- Top regions by EV adoption
SELECT * FROM PON_EV_LAB.CURATED.EV_BY_REGION ORDER BY ev_percentage DESC LIMIT 10;

In [ ]:
-- Infrastructure gaps: high EVs per charging point = expansion opportunity
SELECT * FROM PON_EV_LAB.ANALYTICS.EV_INFRASTRUCTURE_CORRELATION
ORDER BY evs_per_charging_point DESC LIMIT 10;

**Checkpoint — Module 3 complete.** Run `SHOW DYNAMIC TABLES IN DATABASE PON_EV_LAB` and confirm you see **10 Dynamic Tables** — 3 in `CURATED`, 7 in `ANALYTICS`. All should show status `ACTIVE` or `SCHEDULED`.

The query above shows which postal areas have the most EVs per charging point — these are the **infrastructure gaps** where Pon's dealer network could recommend charging station installations. This is the core insight the use case is designed to surface.

---
## Module 4: Scaling & Cost Control

A common objection in enterprises: *"If we give everyone access, won't costs explode?"* Snowflake's answer is architectural — separate compute from storage, then govern the compute.

**Multi-cluster warehouses** auto-scale from 1 to 3 clusters when concurrent users spike (e.g., 20 dealers running dashboards at 9 AM). When load drops, clusters scale back to 1 within minutes.

**Resource monitors** act as hard budget limits. The configuration below caps spend at 100 credits/month and suspends the warehouse at 100% — no surprise bills, ever.

> **Edition note:** Multi-cluster warehouses require **Enterprise edition or higher**. If your trial account is Standard edition, skip the next cell (the ALTER WAREHOUSE for multi-cluster) — the resource monitor and everything else still works.

In [ ]:
-- Enable multi-cluster: auto-scale from 1 to 3 clusters under load
-- NOTE: Requires Enterprise edition. On Standard edition this will error — safe to skip.
ALTER WAREHOUSE PON_ANALYTICS_WH SET
    MIN_CLUSTER_COUNT = 1
    MAX_CLUSTER_COUNT = 3
    SCALING_POLICY = 'STANDARD';

In [ ]:
-- Hard budget limit: 100 credits/month, auto-suspend at 100%
CREATE OR REPLACE RESOURCE MONITOR PON_LAB_MONITOR
    WITH CREDIT_QUOTA = 100
    FREQUENCY = MONTHLY
    START_TIMESTAMP = IMMEDIATELY
    TRIGGERS
        ON 50 PERCENT DO NOTIFY
        ON 75 PERCENT DO NOTIFY
        ON 90 PERCENT DO NOTIFY
        ON 100 PERCENT DO SUSPEND
        ON 110 PERCENT DO SUSPEND_IMMEDIATE;

ALTER WAREHOUSE PON_ANALYTICS_WH SET RESOURCE_MONITOR = PON_LAB_MONITOR;

In [ ]:
SHOW WAREHOUSES LIKE 'PON_ANALYTICS_WH';
-- Verify: Size=SMALL, Resource Monitor=PON_LAB_MONITOR
-- On Enterprise: Min/Max Clusters=1/3. On Standard: clusters will show 1/1 (single cluster)

---
## Module 5: Secure Data Sharing

Today, Pon shares data with its dealer network via CSV exports emailed monthly. The data is stale before it arrives, there's no access control once the file leaves, and nobody knows which version a dealer is using.

**Secure Data Sharing** eliminates all of this. We create a `SHARE` object, grant access to specific analytics tables, and any consumer account sees **live data** — the same bytes, in real-time, with zero copy. If a dealer's contract ends, you revoke the share in one command.

> No data movement. No storage duplication. No stale exports. The dealer sees the same `EV_INFRASTRUCTURE_CORRELATION` table you do, refreshed by Dynamic Tables every hour.

In [ ]:
CREATE OR REPLACE SHARE PON_DEALER_SHARE
    COMMENT = 'EV analytics data for Pon dealer network - live data, no copies';

GRANT USAGE ON DATABASE PON_EV_LAB TO SHARE PON_DEALER_SHARE;
GRANT USAGE ON SCHEMA PON_EV_LAB.ANALYTICS TO SHARE PON_DEALER_SHARE;

GRANT SELECT ON TABLE PON_EV_LAB.ANALYTICS.EV_GROWTH_TRENDS TO SHARE PON_DEALER_SHARE;
GRANT SELECT ON TABLE PON_EV_LAB.ANALYTICS.EV_YOY_GROWTH TO SHARE PON_DEALER_SHARE;
GRANT SELECT ON TABLE PON_EV_LAB.ANALYTICS.EV_INFRASTRUCTURE_CORRELATION TO SHARE PON_DEALER_SHARE;

In [ ]:
-- Verify: should show OUTBOUND share with 3 objects
DESCRIBE SHARE PON_DEALER_SHARE;

---
## Module 6: Marketplace Data Enrichment

So far, everything has been internal RDW data. But the real power comes from **combining your data with external datasets you didn't have to collect**.

The Snowflake Marketplace provides curated, live datasets from third-party providers. We'll acquire two free datasets and join them with our EV analytics — no ETL, no API integration, no data engineering effort:

| Dataset | Provider | What It Adds |
|---|---|---|
| **Dutch Weather Data (KNMI)** | DDBM B.V. | 23M+ hourly weather observations — does cold weather slow EV adoption? |
| **Snowflake Public Data (Free)** | Snowflake | CO2 emissions by country — is the EV transition reducing transport emissions? |

**Before running the cells below**, acquire these datasets:
1. Go to **Data Products → Marketplace**, search for **Dutch Weather Data (KNMI)** by DDBM B.V., click **Get**, accept terms, use database name `DUTCH_WEATHER_DATA_KNMI`
2. Search for **Snowflake Public Data (Free)** by Snowflake, click **Get**, accept terms, use database name `SNOWFLAKE_PUBLIC_DATA_FREE`

> This is the "aha moment" — in 2 clicks you've added 70+ years of weather data and global emissions data to your analytics, queryable in the same SQL alongside your own tables.

In [ ]:
-- Dutch weather: 23M+ hourly observations from 123 stations since 1951
SELECT COUNT(*) AS total_observations
FROM DUTCH_WEATHER_DATA_KNMI.PUBLIC.KNMI_HOURLY_IN_SITU_METEOROLOGICAL_OBSERVATIONS_VALIDATED;

In [ ]:
-- Average temperature and freezing hours by year
SELECT
    YEAR(TIME) AS year,
    ROUND(AVG(TEMP), 1) AS avg_temp_celsius,
    COUNT(CASE WHEN TEMP < 0 THEN 1 END) AS freezing_hours
FROM DUTCH_WEATHER_DATA_KNMI.PUBLIC.KNMI_HOURLY_IN_SITU_METEOROLOGICAL_OBSERVATIONS_VALIDATED
WHERE YEAR(TIME) >= 2015
GROUP BY YEAR(TIME)
ORDER BY year;

In [ ]:
-- Netherlands transport CO2 emissions over time
SELECT
    YEAR(DATE) AS year,
    ROUND(VALUE / 1000000, 2) AS transport_co2_mt
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.CLIMATE_WATCH_TIMESERIES
WHERE GEO_ID = 'country/NLD'
  AND VARIABLE = 'transportation_co2_climate_watch'
ORDER BY year DESC
LIMIT 15;

### Enrichment Views

The exploratory queries above confirm the data is there and meaningful. Now let's create **persistent views** that join marketplace data with our EV analytics. These views live in our database but query the marketplace data in place — no copies, always fresh.

In [ ]:
-- Weather summary view
CREATE OR REPLACE VIEW PON_EV_LAB.CURATED.NL_WEATHER_YEARLY AS
SELECT
    YEAR(TIME) AS year,
    ROUND(AVG(TEMP), 1) AS avg_temp_celsius,
    COUNT(CASE WHEN TEMP < 0 THEN 1 END) AS freezing_hours,
    ROUND(SUM(COALESCE(PRECIP_HOUR, 0)), 0) AS total_precip_mm
FROM DUTCH_WEATHER_DATA_KNMI.PUBLIC.KNMI_HOURLY_IN_SITU_METEOROLOGICAL_OBSERVATIONS_VALIDATED
GROUP BY YEAR(TIME);

-- Transport emissions view
CREATE OR REPLACE VIEW PON_EV_LAB.CURATED.NL_TRANSPORT_EMISSIONS AS
SELECT
    YEAR(DATE) AS year,
    ROUND(VALUE / 1000000, 1) AS transport_co2_mt
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.CLIMATE_WATCH_TIMESERIES
WHERE GEO_ID = 'country/NLD'
  AND VARIABLE = 'transportation_co2_climate_watch'
  AND YEAR(DATE) >= 2010
ORDER BY year;

-- Total emissions view
CREATE OR REPLACE VIEW PON_EV_LAB.CURATED.NL_TOTAL_EMISSIONS AS
SELECT
    YEAR(DATE) AS year,
    ROUND(VALUE / 1000000, 1) AS total_co2_mt
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.CLIMATE_WATCH_TIMESERIES
WHERE GEO_ID = 'country/NLD'
  AND VARIABLE = 'total_excluding_lulucf_co2_climate_watch'
  AND YEAR(DATE) >= 2010
ORDER BY year;

-- Monthly weather patterns
CREATE OR REPLACE VIEW PON_EV_LAB.CURATED.NL_MONTHLY_WEATHER AS
SELECT
    MONTH(TIME) AS month,
    ROUND(AVG(TEMP), 1) AS avg_temp_c,
    ROUND(COUNT(CASE WHEN TEMP < 0 THEN 1 END) * 100.0 / COUNT(*), 1) AS pct_freezing,
    ROUND(AVG(COALESCE(PRECIP_HOUR, 0)) * 30, 0) AS avg_monthly_precip_mm
FROM DUTCH_WEATHER_DATA_KNMI.PUBLIC.KNMI_HOURLY_IN_SITU_METEOROLOGICAL_OBSERVATIONS_VALIDATED
WHERE YEAR(TIME) >= 2020
GROUP BY MONTH(TIME);

In [ ]:
-- Combined view: EV growth + weather + emissions
CREATE OR REPLACE VIEW PON_EV_LAB.ANALYTICS.EV_WEATHER_EMISSIONS AS
SELECT
    e.registration_year AS year,
    e.ev_count AS total_evs,
    e.yoy_growth_percent,
    w.avg_temp_celsius,
    w.freezing_hours,
    t.transport_co2_mt
FROM PON_EV_LAB.ANALYTICS.EV_YOY_GROWTH e
LEFT JOIN PON_EV_LAB.CURATED.NL_WEATHER_YEARLY w ON e.registration_year = w.year
LEFT JOIN PON_EV_LAB.CURATED.NL_TRANSPORT_EMISSIONS t ON e.registration_year = t.year
WHERE e.registration_year >= 2015
ORDER BY e.registration_year;

In [ ]:
-- Regional weather vs EV adoption correlation
CREATE OR REPLACE VIEW PON_EV_LAB.ANALYTICS.REGIONAL_WEATHER_EV_CORRELATION AS
WITH weather_by_region AS (
    SELECT
        CASE
            WHEN STATIONNAME LIKE '%SCHIPHOL%' OR STATIONNAME LIKE '%AMSTERDAM%'
                 OR STATIONNAME LIKE '%ROTTERDAM%' OR STATIONNAME LIKE '%DEN HAAG%'
                 OR STATIONNAME LIKE '%UTRECHT%' OR STATIONNAME LIKE '%HAARLEM%'
                 OR STATIONNAME LIKE '%VALKENBURG%' OR STATIONNAME LIKE '%VOORSCHOTEN%' THEN 'Randstad'
            WHEN STATIONNAME LIKE '%GRONINGEN%' OR STATIONNAME LIKE '%LEEUWARDEN%'
                 OR STATIONNAME LIKE '%EELDE%' OR STATIONNAME LIKE '%HOOGEVEEN%' THEN 'Noord'
            WHEN STATIONNAME LIKE '%MAASTRICHT%' OR STATIONNAME LIKE '%EINDHOVEN%'
                 OR STATIONNAME LIKE '%VOLKEL%' OR STATIONNAME LIKE '%ARCEN%' THEN 'Zuid'
            ELSE 'Overig'
        END AS climate_region,
        ROUND(AVG(TEMP), 1) AS avg_temp,
        COUNT(CASE WHEN TEMP < 0 THEN 1 END) AS total_freezing_hours,
        COUNT(CASE WHEN TEMP < -5 THEN 1 END) AS total_extreme_cold_hours
    FROM DUTCH_WEATHER_DATA_KNMI.PUBLIC.KNMI_HOURLY_IN_SITU_METEOROLOGICAL_OBSERVATIONS_VALIDATED
    WHERE YEAR(TIME) >= 2020
    GROUP BY 1
),
ev_by_region AS (
    SELECT
        CASE
            WHEN postal_area IN ('10','11','12','13','14','15','16','17','18','19',
                                 '20','21','22','23','24','25','26','27','28','29',
                                 '30','31','32','33','34','35','36','37','38','39') THEN 'Randstad'
            WHEN postal_area IN ('88','89','90','91','92','93','94','95','96','97','98','99') THEN 'Noord'
            WHEN postal_area IN ('50','51','52','53','54','55','56','57','58','59',
                                 '60','61','62','63','64','65','66') THEN 'Zuid'
            ELSE 'Overig'
        END AS region,
        SUM(electric_vehicles) AS evs,
        SUM(total_vehicles) AS total,
        ROUND(SUM(electric_vehicles) * 100.0 / NULLIF(SUM(total_vehicles), 0), 2) AS ev_share_pct
    FROM PON_EV_LAB.CURATED.EV_BY_REGION
    GROUP BY 1
)
SELECT
    e.region, e.evs, e.total, e.ev_share_pct,
    w.avg_temp, w.total_freezing_hours, w.total_extreme_cold_hours
FROM weather_by_region w
LEFT JOIN ev_by_region e ON w.climate_region = e.region
ORDER BY ev_share_pct DESC;

In [ ]:
-- EV growth correlated with weather and emissions
SELECT * FROM PON_EV_LAB.ANALYTICS.EV_WEATHER_EMISSIONS;

---
## Final Results

Everything is built. Let's query the analytics layer to answer the original business questions:

1. **How is EV adoption progressing nationally?** → `NATIONAL_EV_SUMMARY`
2. **Is growth accelerating year over year?** → `EV_YOY_GROWTH`
3. **Which postal codes have the most charging points?** → `LAADPALEN_PER_POSTCODE`
4. **Where does charging infrastructure lag behind EV demand?** → `EV_INFRASTRUCTURE_CORRELATION`

These are the same tables the dealer share exposes — what you see here is what the dealers see, live.

In [ ]:
-- National EV summary
SELECT * FROM PON_EV_LAB.ANALYTICS.NATIONAL_EV_SUMMARY;

In [ ]:
-- Year-over-year EV growth
SELECT * FROM PON_EV_LAB.ANALYTICS.EV_YOY_GROWTH ORDER BY registration_year;

In [ ]:
-- Target model: Laadpalen per postcode
SELECT * FROM PON_EV_LAB.ANALYTICS.LAADPALEN_PER_POSTCODE ORDER BY aantal DESC LIMIT 20;

In [ ]:
-- The answer: EV adoption vs charging infrastructure by region
-- High evs_per_charging_point = regions where infrastructure lags behind demand
SELECT * FROM PON_EV_LAB.ANALYTICS.EV_INFRASTRUCTURE_CORRELATION
ORDER BY evs_per_charging_point DESC;

In [ ]:
-- All Dynamic Tables and their refresh status
SHOW DYNAMIC TABLES IN DATABASE PON_EV_LAB;

---
## What You Built

In ~90 minutes you've gone from zero to a production-grade data platform:

| Component | What | Snowflake Feature |
|---|---|---|
| **5 raw tables** | Live RDW government data | External Access Integration + Python UDF |
| **10 Dynamic Tables** | Auto-refreshing pipeline, bronze → silver → gold | `TARGET_LAG = '1 hour'` — no scheduler needed |
| **Multi-cluster warehouse** | Concurrent user scaling (1→3 clusters) | Enterprise auto-scaling |
| **Resource monitor** | Hard budget limit: 100 credits/month | Auto-suspend at threshold |
| **Secure data share** | Live dealer access, zero-copy | `CREATE SHARE` — revocable in seconds |
| **6 enrichment views** | Weather + emissions context | Marketplace — 2 clicks, no ETL |

### The Bigger Picture

This lab demonstrates the pattern Pon can use across all its brands — not just for EV analytics, but for any data engineering workload:
- **APIs as first-class data sources** — no middleware required
- **Declarative pipelines** — write the "what", Snowflake handles the "when"
- **Governed sharing** — data stays in one place, access is controlled centrally
- **Marketplace enrichment** — external data joins your internal data with zero integration effort

### Next Steps
- Deploy the **Streamlit dashboard** (`streamlit_app.py`) for interactive visualization
- Explore the **lab guide** (`lab_guide.md`) for governance patterns, production deployment options, and the bonus Cortex AI module
- Consider loading the full 16.7M-row datasets for production use (remove the `$limit` parameter from the API calls)